# BudgetSwitch W3 — Cold-Start SFT

**Mục tiêu:** Train tool-use LoRA (rank 16) trên gold trajectories để Qwen2.5-3B học Vietnamese tool-calling format. Sau SFT, success rate target: 50-70% trên easy steps.

**Tại sao cần SFT trước GRPO:**
- Base model chưa biết Vietnamese tool-calling format (pilot cho thấy 12%)
- SFT dạy model cú pháp mode tokens + JSON tool call (syntactic warmup)
- SFT dùng random mode sampling — **không phải oracle mode labels** (Critical Rule 1)
- Sau SFT, GRPO mới có thể học policy hữu ích

**CRITICAL RULES:**
- Rule 1: SFT data dùng RANDOM mode tokens, không phải oracle/correct mode
- Rule 2: Sau SFT, FREEZE tool-use LoRA trước khi add mode-routing LoRA cho GRPO

In [1]:
%%bash
# Cell 1: Sửa đổi ép symlink vào thư mục hệ thống công cộng của Kaggle
echo "=== Kích hoạt Pha Vá Lỗi Hệ Thống libnvJitLink ==="

# Tìm chính xác vị trí file gốc CUDA 12.8 đang có trên Kaggle
LIB_SRC=$(find /usr/local/cuda-12.8/ -name "libnvJitLink.so.12" 2>/dev/null | head -1)

if [ -z "$LIB_SRC" ]; then
    echo "⚠️ Không tìm thấy thư mục gốc CUDA 12.8, đang tìm bản CUDA dự phòng..."
    LIB_SRC=$(find /usr/local/ -name "libnvJitLink.so.12" 2>/dev/null | head -1)
fi

if [ -n "$LIB_SRC" ]; then
    echo "Tìm thấy file gốc tại: $LIB_SRC"
    
    # Ép liên kết mềm thẳng vào thư mục chứa thư viện hệ thống công cộng của Ubuntu
    ln -sf "$LIB_SRC" /usr/lib/x86_64-linux-gnu/libnvJitLink.so.13
    ln -sf "$LIB_SRC" /usr/local/lib/libnvJitLink.so.13
    ln -sf "$LIB_SRC" /usr/local/cuda/lib64/libnvJitLink.so.13
    
    # Chạy ldconfig với quyền cập nhật để bộ nạp Linux nhận diện ngay lập tức
    ldconfig
    echo "✅ Đã vá cứng thành công liên kết libnvJitLink.so.13 hệ thống!"
else
    echo "❌ LỖI NGHIÊM TRỌNG: Không tìm thấy bất kỳ file libnvJitLink.so nào để tạo liên kết."
fi

=== Kích hoạt Pha Vá Lỗi Hệ Thống libnvJitLink ===
Tìm thấy file gốc tại: /usr/local/cuda-12.8/targets/x86_64-linux/lib/libnvJitLink.so.12
✅ Đã vá cứng thành công liên kết libnvJitLink.so.13 hệ thống!


/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/li

In [2]:
# Cell 2: Install
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "vllm", "torchvision", "torchaudio"], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"],
               capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "--upgrade", "peft", "accelerate"], capture_output=True)
print("✅ Install complete")

✅ Install complete


In [3]:
# Cell 3: Setup
import os, sys, json, random
from pathlib import Path

REPO = Path("/kaggle/working/budgetswitch")
if not REPO.exists():
    import subprocess
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    subprocess.run(["git", "clone",
                    f"https://{token}@github.com/ductaip/budgetswitch.git",
                    str(REPO)], check=True)
    print("✅ Repo cloned")
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], capture_output=True)
    print("✅ Repo updated")

sys.path.insert(0, str(REPO))

with open(REPO / "data/vitool/tool_catalog.json") as f:
    CATALOG_RAW = json.load(f)
CATALOG = {k: v for k, v in CATALOG_RAW.items() if not k.startswith("_")}
print(f"✅ Catalog loaded: {len(CATALOG)} tools")

✅ Repo updated
✅ Catalog loaded: 36 tools


In [4]:
# Cell 4: Load model + add tool-use LoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    fast_inference=False,
    dtype=None,
)

# Tool-use LoRA (rank 16) — this is what SFT trains
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"✅ Model + tool-use LoRA ready")
print(f"   Trainable: {trainable:,} ({100*trainable/total:.3f}%)")
print(f"   Peak VRAM: {peak:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0526 15:45:44.194000 459 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0526 15:45:44.227000 459 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ Model + tool-use LoRA ready
   Trainable: 29,933,568 (1.636%)
   Peak VRAM: 2.52 GB


In [5]:
# Cell 5: Build SFT dataset from gold trajectories
#
# CRITICAL RULE 1: mode tokens are sampled RANDOMLY — not oracle/correct mode.
# SFT teaches model the FORMAT of (mode_token + JSON tool call).
# GRPO will later teach WHICH mode to pick (that's the RL part).
#
from data.vitool.schema import load_trajectories
from modes.m0_fast.m0_fast import _build_m0_prompt

random.seed(42)
MODES = ["M0", "M1", "M2", "M3"]

# Load all gold trajectories
GOLD_PATH = str(REPO / "data/vitool/vitool_all.jsonl")
trajectories = load_trajectories(GOLD_PATH)
print(f"Loaded {len(trajectories)} gold trajectories")

def build_sft_example(traj, step_idx):
    """
    Build one SFT training example.
    
    Format:
      <prompt> → mode_token + JSON tool call
    
    Mode is sampled RANDOMLY (not oracle) — see Critical Rule 1.
    """
    gt = traj.steps[step_idx]
    history = [
        ({"tool_name": s.expected_tool, "params": s.expected_params}, {})
        for s in traj.steps[:step_idx]
    ]
    
    # Build prompt using M0 (same format for all modes — mode token prepended)
    prompt = _build_m0_prompt(traj.query, history, CATALOG)
    
    # Random mode token (CRITICAL: not oracle mode)
    mode_token = random.choice(MODES)
    
    # Target: mode token + correct tool call
    target = (
        f"{mode_token}\n"
        f'{{"tool_name": "{gt.expected_tool}", '
        f'"params": {json.dumps(gt.expected_params, ensure_ascii=False)}}}'
    )
    
    return {"prompt": prompt, "completion": target}

# Build dataset
sft_examples = []
for traj in trajectories:
    for step_idx in range(traj.num_steps):
        ex = build_sft_example(traj, step_idx)
        sft_examples.append(ex)

print(f"✅ SFT dataset: {len(sft_examples)} examples")
print(f"   Mode distribution (random): should be ~25% each")
from collections import Counter
modes_used = Counter(ex["completion"][0:2] for ex in sft_examples)
print(f"   {dict(modes_used)}")

# Sample
ex = sft_examples[0]
print(f"\nSample prompt (last 200 chars): ...{ex['prompt'][-200:]}")
print(f"Sample completion: {ex['completion'][:100]}")

Loaded 200 gold trajectories
✅ SFT dataset: 239 examples
   Mode distribution (random): should be ~25% each
   {'M0': 70, 'M2': 47, 'M1': 67, 'M3': 55}

Sample prompt (last 200 chars): ...culate: Perform numerical calculation
- wiki_lookup: Look up factual info<|im_end|>
<|im_start|>user
Xin chào, tôi muốn kiểm tra số dư tài khoản của mình là 1234567890<|im_end|>
<|im_start|>assistant

Sample completion: M0
{"tool_name": "check_balance", "params": {"account_number": "1234567890"}}


In [6]:
# Cell 6: Format as HuggingFace dataset
from datasets import Dataset

def format_for_unsloth(example):
    """Format as conversation for Unsloth SFT."""
    return {
        "text": (
            example["prompt"] +
            example["completion"] +
            tokenizer.eos_token
        )
    }

hf_dataset = Dataset.from_list(sft_examples).map(format_for_unsloth)

# Train/val split (90/10)
split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds = split["test"]

print(f"✅ Dataset ready")
print(f"   Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"   Sample text length: {len(hf_dataset[0]['text'])} chars")

Map:   0%|          | 0/239 [00:00<?, ? examples/s]

✅ Dataset ready
   Train: 215 | Val: 24
   Sample text length: 1896 chars


In [7]:
# ==========================================
# CELL 7: SFT TRAINING (BẢN SỬA LỖI)
# ==========================================
from trl import SFTTrainer, SFTConfig

# 1. Định nghĩa cấu hình huấn luyện
config = SFTConfig(
    output_dir = "/kaggle/working/sft_checkpoints",
    num_train_epochs = 3,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4, # effective batch = 16
    warmup_steps = 10,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    eval_strategy = "epoch",  # Đã đồng bộ theo bản sửa lỗi trước
    save_strategy = "epoch",
    load_best_model_at_end = True,
    optim = "adamw_torch", 
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    report_to = "none",
    dataset_text_field = "text",
    max_seq_length = 1024,
)

print(f"Starting SFT training...")
print(f"  Epochs: 3 | Batch: 4 × 4 accum = 16 effective")
print(f"  Train examples: {len(train_ds)}")
print(f"  Estimated: ~10-15 mins on T4 (Unsloth optimized)")
print()

# ============================================================
# 👉 DÒNG FIX CHÍ MẠNG: Khởi tạo thực thể trainer còn thiếu ở đây
# ============================================================
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    dataset_text_field = "text",
    max_seq_length = 1024,
    dataset_num_proc = 2,
    packing = False, # Không bật packing để giữ nguyên cấu trúc mode token đầu chuỗi
    args = config,
)

# 2. Kích hoạt tiến trình huấn luyện LoRA
trainer_stats = trainer.train()

print(f"\n✅ SFT done!")
print(f"   Train loss: {trainer_stats.training_loss:.4f}")
print(f"   Runtime: {trainer_stats.metrics['train_runtime'] / 60:.1f} min")

Starting SFT training...
  Epochs: 3 | Batch: 4 × 4 accum = 16 effective
  Train examples: 215
  Estimated: ~10-15 mins on T4 (Unsloth optimized)



Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=8):   0%|          | 0/215 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=8):   0%|          | 0/24 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 215 | Num Epochs = 3 | Total steps = 42
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,1.298106,0.247798
2,0.243505,0.087024
3,0.074478,0.086357


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/sft_checkpoints/checkpoint-14/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/sft_checkpoints/checkpoint-28/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/sft_checkpoints/checkpoint-42/tokenizer_config.json.



✅ SFT done!
   Train loss: 0.4176
   Runtime: 11.9 min


In [8]:
# Cell 8: Save tool-use LoRA adapter
ADAPTER_PATH = "/kaggle/working/tool_use_lora"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"✅ Tool-use LoRA saved to {ADAPTER_PATH}")

# Check adapter size
total_bytes = sum(f.stat().st_size for f in Path(ADAPTER_PATH).rglob("*") if f.is_file())
print(f"   Adapter size: {total_bytes/1e6:.1f} MB")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/tool_use_lora/tokenizer_config.json.


✅ Tool-use LoRA saved to /kaggle/working/tool_use_lora
   Adapter size: 131.2 MB


In [9]:
# Cell 9: Quick sanity check — success rate should be MUCH better than base model
# (target: ≥40% on easy steps, up from 12%)
import time
from modes.m0_fast.m0_fast import _build_m0_prompt, _parse_tool_call
from verifiers.universal.schema_verifier import verify_step_outcome
from data.vitool.schema import load_trajectories

FastLanguageModel.for_inference(model)

# Test on 20 easy steps from pilot set
PILOT_PATH = str(REPO / "data/vitool/vitool_pilot.jsonl")
pilot_trajs = load_trajectories(PILOT_PATH)
easy_steps = [
    (traj, i)
    for traj in pilot_trajs
    for i, s in enumerate(traj.steps)
    if s.difficulty.value == "easy"
][:20]

print(f"Testing on {len(easy_steps)} easy steps...")
success_m0 = 0
for traj, step_idx in easy_steps:
    gt = traj.steps[step_idx]
    history = [({"tool_name": s.expected_tool, "params": s.expected_params}, {})
               for s in traj.steps[:step_idx]]
    prompt = _build_m0_prompt(traj.query, history, CATALOG)
    
    import torch
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=150, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    step_out = _parse_tool_call(raw, CATALOG)
    
    gt_dict = {"expected_tool": gt.expected_tool,
               "expected_params": gt.expected_params,
               "fuzzy_keys": gt.fuzzy_keys}
    if verify_step_outcome(step_out, gt_dict, CATALOG):
        success_m0 += 1

print(f"\n✅ Post-SFT M0 success on easy steps: {success_m0}/{len(easy_steps)} ({100*success_m0/len(easy_steps):.0f}%)")
print(f"   Pre-SFT baseline: 12%")
print(f"   Target: ≥40%")
if success_m0 / len(easy_steps) >= 0.40:
    print("\n🟢 SFT successful — proceed to W3 mode-routing LoRA setup")
elif success_m0 / len(easy_steps) >= 0.25:
    print("\n🟡 SFT partial — consider 1 more epoch or larger batch")
else:
    print("\n🔴 SFT weak — check training loss, possible data format issue")

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Testing on 20 easy steps...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1


✅ Post-SFT M0 success on easy steps: 18/20 (90%)
   Pre-SFT baseline: 12%
   Target: ≥40%

🟢 SFT successful — proceed to W3 mode-routing LoRA setup


## Sau khi SFT xong

**Download và push lên git:**
```bash
# Download từ Kaggle output:
# /kaggle/working/tool_use_lora/  (toàn bộ folder)

# Push lên git
git add models/tool_use_lora/
git commit -m "W3: cold-start SFT tool-use LoRA (rank 16, 3 epochs)"
git push
```

**Nếu Cell 9 ra ≥40% (🟢):** Proceed W3 Part 2 — add mode-routing LoRA (rank 8) và setup GRPO.

**Nếu 25-40% (🟡):** Chạy thêm 1 epoch:
```python
trainer.train(resume_from_checkpoint=True)
```

**Nếu <25% (🔴):** Check Cell 7 training loss — nếu loss không giảm, data format có vấn đề. Báo lại.